# Drop-in with scikit-learn

`HMMClassifier` is a scikit-learn-compatible wrapper around `hmm_core.fit`. Use it in any sklearn workflow : `Pipeline`, `GridSearchCV`, `cross_val_score`, etc.

This is **Phase I.2** of the hybrid distribution strategy (ADR-0012) — hmm-studio slots into your existing sklearn-based research workflow with zero environment change.

## Basic usage : fit / predict

In [ ]:
import numpy as np
from hmm_core.sklearn_compat import HMMClassifier

rng = np.random.default_rng(42)
X = np.concatenate([
    rng.normal(0.0, 0.5, (80, 1)),
    rng.normal(5.0, 0.5, (80, 1)),
    rng.normal(0.0, 0.5, (80, 1)),
    rng.normal(5.0, 0.5, (80, 1)),
])
y_true = np.concatenate([np.zeros(80), np.ones(80), np.zeros(80), np.ones(80)]).astype(int)

clf = HMMClassifier(n_states=2, n_iter=50)
clf

In [ ]:
clf.fit(X)
clf     # rich HTML view of the fitted model

In [ ]:
predictions = clf.predict(X)
predictions[:30]

## Supervised mode : pass `y` to `fit`

When labels are available, supervised closed-form MLE is used (no EM iterations).

In [ ]:
clf_sup = HMMClassifier(n_states=2)
clf_sup.fit(X, y_true)
print(f"Converged in {clf_sup.n_iter_} iteration (supervised = one-pass)")
print(f"Accuracy on training data : {clf_sup.score(X, y_true):.3f}")

## Use in a sklearn Pipeline

Chain HMM with any standard sklearn preprocessing.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("hmm", HMMClassifier(n_states=2, n_iter=20)),
])
pipe.fit(X, y_true)
print(f"Pipeline accuracy : {pipe.score(X, y_true):.3f}")

## Grid search over K (model selection)

Use `GridSearchCV` to search the right number of states. With a true K=2 problem, GridSearchCV should pick K=2.

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "n_states": [2, 3, 4],
    "init_strategy": ["kmeans", "random"],
}
search = GridSearchCV(
    HMMClassifier(n_iter=20),
    param_grid,
    cv=2,
    scoring="accuracy",
    n_jobs=1,
)
search.fit(X, y_true)
print(f"Best params : {search.best_params_}")
print(f"Best score  : {search.best_score_:.3f}")

## Time-series cross-validation

HMMs assume sequential data, so use `TimeSeriesSplit` rather than `KFold`.

In [ ]:
from sklearn.model_selection import TimeSeriesSplit, cross_val_score

scores = cross_val_score(
    HMMClassifier(n_states=2, n_iter=20),
    X, y_true,
    cv=TimeSeriesSplit(n_splits=3),
    scoring="accuracy",
)
print(f"CV scores : {scores}")
print(f"Mean ± std : {scores.mean():.3f} ± {scores.std():.3f}")

## Inspect fitted attributes

All sklearn conventions : attributes ending with `_` are populated after fit.

In [ ]:
clf = HMMClassifier(n_states=2, n_iter=30).fit(X)

print(f"transmat_       : shape {clf.transmat_.shape}")
print(f"startprob_      : {clf.startprob_}")
print(f"classes_        : {clf.classes_}")
print(f"n_iter_         : {clf.n_iter_}")
print(f"log_likelihood_ : {clf.log_likelihood_:.3f}")
print(f"bic_            : {clf.bic_:.3f}")
print(f"aic_            : {clf.aic_:.3f}")
print(f"converged_      : {clf.converged_}")
print(f"n_features_in_  : {clf.n_features_in_}")

## What this unlocks

- **Existing sklearn workflows** integrate hmm-studio without changes
- **Grid search** over HMM hyperparameters (K, topology, init strategy)
- **Cross-validation** with proper temporal splitting
- **Pipelines** with sklearn preprocessing + hmm-studio modeling
- **Persistence** via `joblib.dump(clf, 'model.joblib')` (sklearn convention)

Note : HMMClassifier wraps `hmm_core.fit.fit` which uses standard Baum-Welch with optional state labels for supervised mode. For covariate-dependent transitions (NHMM), GMM emissions per state (GMM-NHMM), or parallel chains (Factorial NHMM), use the direct APIs in `hmm_core.nhmm`, `hmm_core.gmm_nhmm`, `hmm_core.factorial_nhmm`. A sklearn wrapper for NHMM is on the roadmap as Phase I.2.1.